# 1.5 — Random Forest + export para o ESP32 (Forma 1, InfluxDB)

Treina um classificador **normal × anomalia** com as 7 features lidas do **InfluxDB Cloud**
(gravadas pelo Node-RED a partir do `app17-9`).

Usamos **Random Forest + StandardScaler** (e não uma rede neural) porque o objetivo é, mais
adiante, **embarcar o modelo no ESP32** (AIoT) com a biblioteca **micromlgen** — mesmo
pipeline dos `app21`/`app22`.

> **Atende Sprint 4:** item 1 (base analítica por janela), item 2 (treino/teste + métricas +
> matriz de confusão) e item 3 (**feature importance** + interpretação física).

In [ ]:
!pip install -q influxdb-client pandas scikit-learn matplotlib micromlgen

## 1) Ler as features do InfluxDB Cloud

Preencha com os seus valores (mesmos do nó InfluxDB do Node-RED).

In [ ]:
from influxdb_client import InfluxDBClient
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"
INFLUX_ORG    = "SUA_ORG"
INFLUX_BUCKET = "sensores"
MEASUREMENT   = "vibracao_features"

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)

flux = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -30d)
  |> filter(fn: (r) => r._measurement == "{MEASUREMENT}")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "label",
        "mean_ax", "mean_ay", "mean_az",
        "std_ax", "std_ay", "std_az", "rms_mag"])
  |> sort(columns: ["_time"])
'''
df = client.query_api().query_data_frame(flux)
if isinstance(df, list):
    df = pd.concat(df, ignore_index=True)
df = df.rename(columns={"_time": "time"})
print("Janelas:", len(df)); print(df["label"].value_counts())

## 2) Selecionar features, classes e fazer o split SEM vazamento

- Usamos as **7 features**: `mean_ax/ay/az`, `std_ax/ay/az`, `rms_mag`.
- Classes: `ligado_normal` (0) × `ligado_anomalia` (1). Qualquer `parado` é descartado.
- **Regra de ouro (Aula 14, slide 25):** janelas vizinhas no tempo são quase iguais; um
  `train_test_split` aleatório vazaria informação do treino para o teste. Como aqui só temos
  a tag `label` (sem rodadas numeradas), fazemos um **split cronológico por classe**: os
  primeiros 70% de cada classe vão para treino e os últimos 30% para teste. O ideal mesmo é
  coletar rodadas separadas (`normal_01`, `normal_02`…) e separar por rodada.

In [ ]:
FEATURES = ["mean_ax", "mean_ay", "mean_az", "std_ax", "std_ay", "std_az", "rms_mag"]
CLASSES  = {"ligado_normal": 0, "ligado_anomalia": 1}

df = df[df["label"].isin(CLASSES)].dropna(subset=FEATURES).sort_values("time")
df["y"] = df["label"].map(CLASSES)

train_idx, test_idx = [], []
for lab, g in df.groupby("label"):
    corte = int(len(g) * 0.7)
    train_idx += list(g.index[:corte])
    test_idx  += list(g.index[corte:])

X_train = df.loc[train_idx, FEATURES].values
X_test  = df.loc[test_idx,  FEATURES].values
y_train = df.loc[train_idx, "y"].values
y_test  = df.loc[test_idx,  "y"].values
print(f"Treino: {len(y_train)} janelas | Teste: {len(y_test)} janelas")

## 3) StandardScaler + Random Forest

O `StandardScaler` é **ajustado só no treino** (depois o ESP32 usará os mesmos `means`/`scales`).
A floresta é pequena de propósito (poucas árvores/profundidade) para caber no ESP32.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=42)
clf.fit(X_train_s, y_train)

## 4) Métricas e matriz de confusão

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

y_pred = clf.predict(X_test_s)
print("Acurácia:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, target_names=["normal", "anomalia"]))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["normal", "anomalia"]).plot()
plt.show()

## 5) Feature importance (Sprint 4 – 25 pontos)

O Random Forest tem importância **nativa** (`feature_importances_`). Para reforçar, também
calculamos a **permutation_importance** no conjunto de teste.

In [ ]:
from sklearn.inspection import permutation_importance

imp_nativa = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
perm = permutation_importance(clf, X_test_s, y_test, n_repeats=20, random_state=42)
imp_perm = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp_nativa.plot.barh(ax=axes[0], color="tab:blue");  axes[0].set_title("Importância nativa (RF)")
imp_perm.plot.barh(ax=axes[1], color="tab:orange");  axes[1].set_title("Permutation importance (teste)")
plt.tight_layout(); plt.show()

### Interpretação física

Espera-se que **`rms_mag`** e os **`std_*`** liderem: vibração eleva a intensidade efetiva
(RMS) e a variabilidade (desvio padrão) do sinal, enquanto a **média** quase não muda (a
oscilação positiva e negativa se cancela — Aula 14, slides 12–13). Ou seja, o modelo aprende
exatamente o que a física diz: *anomalia = mais energia/variação na janela*.

## 6) Exportar para o ESP32 (micromlgen)

Gera dois headers C++ (mesmo padrão do `app22`):
- `AIoTVibracaoRF_micromlgen.hpp` — a floresta (`modeloRF.predict(...)` → 0/1).
- `AIoTVibracaoScaler.hpp` — os `means`/`scales` do StandardScaler (7 features).

No Colab eles são baixados automaticamente. O firmware embarcado em si fica para depois.

In [ ]:
from micromlgen import port

with open("AIoTVibracaoRF_micromlgen.hpp", "w") as f:
    f.write(port(clf))

def gerar_scaler_hpp(scaler, n):
    means  = ", ".join(f"{m:.10f}f" for m in scaler.mean_)
    scales = ", ".join(f"{s:.10f}f" for s in scaler.scale_)
    return f'''#ifndef STANDARD_SCALER_HPP
#define STANDARD_SCALER_HPP
// StandardScaler das 7 features de vibracao: mean_ax/ay/az, std_ax/ay/az, rms_mag
namespace Scaler {{
    const static float means[{n}]  = {{ {means} }};
    const static float scales[{n}] = {{ {scales} }};
    inline void std(const float* input, float* output) {{
        for (int i = 0; i < {n}; i++) output[i] = (input[i] - means[i]) / scales[i];
    }}
}}
#endif
'''

with open("AIoTVibracaoScaler.hpp", "w") as f:
    f.write(gerar_scaler_hpp(scaler, len(FEATURES)))

try:
    from google.colab import files
    files.download("AIoTVibracaoRF_micromlgen.hpp")
    files.download("AIoTVibracaoScaler.hpp")
except Exception:
    print("Arquivos gerados na pasta atual (fora do Colab).")

> **Dica (igual aos app21/app22):** se o `.hpp` da floresta não compilar no ESP32, reduza
> `n_estimators` (10–20) ou `max_depth` (5–10) e gere de novo. E nunca misture o scaler de uma
> execução com a floresta de outra — os `means`/`scales` precisam casar.